# 🚴 M2_01 Solutions: Amsterdam Bike Data Acquisition

**Complete solutions with explanations**

---

## 📋 How to Use This Notebook

1. **Try first!** Attempt each task in the main notebook before checking solutions
2. **Learn from differences**: Compare your approach with the solution
3. **Understand, don't copy**: Read the explanations and comments

---

## Setup (Same as Main Notebook)

In [ ]:
# Standard library
import os
import sys
import json
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Setup complete!")

## Configuration

In [ ]:
# API Configuration
BASE_URL = "http://api.citybik.es/v2"
NETWORK_ID = "ov-fiets"
NETWORK_URL = f"{BASE_URL}/networks/{NETWORK_ID}"

# File paths
DATA_DIR = Path('../../data/raw') if 'notebooks' in os.getcwd() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output filename with timestamp
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUTPUT_FILE = DATA_DIR / f'amsterdam_bike_{TIMESTAMP}.json'

print("✅ Configuration set")
print(f"🌐 API URL: {NETWORK_URL}")
print(f"💾 Output file: {OUTPUT_FILE}")

---

## ✅ Task 4.1 Solution: Make Your First API Request

**Note:** This corresponds to Part 4 in the main notebook.

In [ ]:
# SOLUTION: Task 4.1 - Fetch bike data

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}\n")

# Make GET request
response = requests.get(NETWORK_URL)

# Check if successful
if response.status_code == 200:
    # Parse JSON
    data = response.json()
    
    # Extract information
    network_name = data['network']['name']
    city = data['network']['location']['city']
    country = data['network']['location']['country']
    num_stations = len(data['network']['stations'])
    
    # Print results
    print(f"✅ Success! Fetched data for:")
    print(f"   Network: {network_name}")
    print(f"   Location: {city}, {country}")
    print(f"   Stations: {num_stations}")
else:
    print(f"❌ Failed: HTTP {response.status_code}")

---

## ✅ Task 4.2 Solution: Explore the JSON Structure

**Note:** This corresponds to Part 4 in the main notebook.

In [ ]:
# SOLUTION: Task 4.2 - Explore JSON structure

# Get first station
first_station = data['network']['stations'][0]

# Pretty print with json.dumps
print("📋 First station structure:")
print(json.dumps(first_station, indent=2))

print("\n" + "="*60)
print("Field types:")
print("="*60)
for key, value in first_station.items():
    print(f"{key:20s} : {type(value).__name__}")

---

## ✅ Task 5.1 Solution: Implement Robust Error Handling

**Note:** This corresponds to Part 5 in the main notebook.

In [ ]:
# SOLUTION: Task 5.1 - Comprehensive error handling

def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with comprehensive error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ov-fiets')
    timeout : int
        Request timeout in seconds (default: 10)
    
    Returns:
    --------
    dict or None
        JSON data if successful, None if any error occurs
    """
    url = f"{BASE_URL}/networks/{network_id}"
    
    try:
        # Make request with timeout
        print(f"📡 Fetching data from: {url}")
        response = requests.get(url, timeout=timeout)
        
        # Raise exception for bad HTTP status (4xx, 5xx)
        response.raise_for_status()
        
        # Parse and return JSON
        data = response.json()
        print(f"✅ Success! Fetched {len(data['network']['stations'])} stations")
        return data
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.ConnectionError:
        print("🔌 Connection Error: Could not connect to API")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error {e.response.status_code}: {e}")
        if e.response.status_code == 404:
            print(f"   Network '{network_id}' not found")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request Error: {e}")
        return None
        
    except json.JSONDecodeError:
        print("📝 JSON Error: Could not parse response")
        return None


# Test the function
print("🧪 Testing error handling function...")
print("=" * 60)

# Test 1: Valid network
print("\nTest 1: Valid network (ov-fiets)")
result = fetch_bike_data_safe(NETWORK_ID, timeout=10)
if result:
    print(f"✅ Test 1 passed!\n")

# Test 2: Deprecated endpoint (real-world example)
print("\nTest 2: Deprecated endpoint (ns-bike)")
print("⚠️ Note: 'ns-bike' is still documented on many sites but was deprecated")
print("   This is a common real-world scenario!")
result = fetch_bike_data_safe("ns-bike", timeout=10)
if result is None:
    print("✅ Test 2 passed - error handling works for deprecated APIs\n")

print("=" * 60)

---

## 📊 Part 7: Convert to DataFrame - Solutions

**Note:** These solutions correspond to Part 7 in the main notebook.

**Important:** The solution names the DataFrame `df_bikes` - this variable is used throughout.

**⚠️ Key Point:** The OV-fiets API endpoint returns stations for **all cities in the Netherlands**, not just Amsterdam. In Task 7.5, we'll filter for Amsterdam-specific stations using the city metadata.

In [ ]:
# Fetch data using our safe function
data = fetch_bike_data_safe(NETWORK_ID)
if not data:
    raise ValueError("Failed to fetch data")

### Task 7.1 Solution: Extract and Convert to DataFrame

In [ ]:
# SOLUTION: Task 7.1 - Extract and convert to DataFrame (named df_bikes)

# Extract stations list
stations = data['network']['stations']

# Convert to DataFrame
df_bikes = pd.DataFrame(stations)

# Display info
print(f"✅ Created DataFrame: {df_bikes.shape[0]} rows × {df_bikes.shape[1]} columns\n")
print("First 3 rows:")
display(df_bikes.head(3))

print("\n" + "="*60)
print("DataFrame Info:")
print("="*60)
df_bikes.info()

### Task 7.2 Solution: Clean Column Names and Parse Timestamps

In [ ]:
# SOLUTION: Task 7.2 - Clean column names and parse timestamps

# Parse timestamp (strip 'Z' first to avoid parsing issues)
df_bikes['timestamp'] = pd.to_datetime(df_bikes['timestamp'].str.rstrip('Z'))

# Rename columns
df_bikes = df_bikes.rename(columns={
    'free_bikes': 'bikes_available',
    'empty_slots': 'docks_available'
})

print("✅ Parsed timestamps and renamed columns")
print(f"\nTimestamp column type: {df_bikes['timestamp'].dtype}")

### Task 7.3 Solution: Add Derived Columns

In [ ]:
# SOLUTION: Task 7.3 - Add derived columns

# Handle missing docks_available (some APIs don't provide this field)
# Replace None with 0 for calculation
df_bikes['docks_available'] = pd.to_numeric(df_bikes['docks_available'], errors='coerce').fillna(0).astype(int)

# Total capacity
df_bikes['total_capacity'] = df_bikes['bikes_available'] + df_bikes['docks_available']

# Utilization percentage (handle division by zero)
df_bikes['utilization_pct'] = (
    (df_bikes['bikes_available'] / df_bikes['total_capacity'] * 100)
    .fillna(0)  # Replace NaN with 0 for stations with 0 capacity
    .round(2)
)

# Empty flag
df_bikes['is_empty'] = df_bikes['bikes_available'] == 0

print("✅ Added derived columns")
print("\nSample values:")
display(df_bikes[['bikes_available', 'docks_available', 'total_capacity', 
                   'utilization_pct', 'is_empty']].head())

### Task 7.4 Solution: Add Network Metadata

In [ ]:
# SOLUTION: Task 7.4 - Add network metadata

df_bikes['network_id'] = data['network']['id']
df_bikes['network_name'] = data['network']['name']
df_bikes['city'] = data['network']['location']['city']
df_bikes['country'] = data['network']['location']['country']

print("✅ Added network metadata")
print(f"\nNetwork: {df_bikes['network_name'].iloc[0]}")
print(f"Location: {df_bikes['city'].iloc[0]}, {df_bikes['country'].iloc[0]}")

### Task 7.5 Solution: Filter for Amsterdam Stations and Validate

**Important:** The OV-fiets network covers all of the Netherlands, not just Amsterdam. We need to filter for stations that have "Amsterdam" in their city name.

In [ ]:
# SOLUTION: Task 7.5 - Filter Amsterdam stations, reorder columns, and validate

# Filter for Amsterdam stations
# Note: OV-fiets API returns stations for all Dutch cities
# We filter by checking if "Amsterdam" appears in the station name
print("🔍 Filtering for Amsterdam stations...")
print(f"Total stations before filtering: {len(df_bikes)}")

# Check unique cities to understand the data
print(f"\nUnique cities in dataset: {df_bikes['city'].nunique()}")
print(f"Sample station names:")
print(df_bikes['name'].head(10).tolist())

# Filter for Amsterdam - check if "Amsterdam" is in the station NAME
# (not the city field, which is set at the network level)
df_bikes = df_bikes[df_bikes['name'].str.contains('Amsterdam', case=False, na=False)].copy()

print(f"\n✅ Filtered to Amsterdam stations: {len(df_bikes)} stations")

# Define column order
column_order = [
    'id', 'name', 'latitude', 'longitude',
    'bikes_available', 'docks_available', 'total_capacity', 'utilization_pct',
    'timestamp', 'network_id', 'network_name', 'city', 'country'
]

# Reorder
df_bikes = df_bikes[column_order]

print("✅ Reordered columns\n")
print("First 10 rows:")
display(df_bikes.head(10))

print("\n" + "="*60)
print("Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

---

## 🔍 Part 8: Data Quality and Exploration - Solutions

**Note:** These solutions correspond to Part 8 in the main notebook.

### Task 8.1 Solution: Calculate Summary Statistics

In [ ]:
# SOLUTION: Task 8.1 - Calculate summary statistics

print("📈 Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

print("\n" + "="*60)
print("🚴 Bike Availability Summary:")
print("="*60)
print(f"Total Stations: {len(df_bikes)}")
print(f"Total Bikes Available: {df_bikes['bikes_available'].sum()}")
print(f"Total Docks Available: {df_bikes['docks_available'].sum()}")
print(f"Average Station Capacity: {df_bikes['total_capacity'].mean():.1f}")
print(f"Average Utilization: {df_bikes['utilization_pct'].mean():.1f}%")

print("\n" + "="*60)
print("⚠️ Problem Stations:")
print("="*60)
empty_stations = df_bikes[df_bikes['bikes_available'] == 0]
full_stations = df_bikes[df_bikes['docks_available'] == 0]
print(f"Empty stations (no bikes): {len(empty_stations)}")
print(f"Full stations (no docks): {len(full_stations)}")

---

### 💡 Critical Domain Insight: OV-fiets System Characteristics

**🔍 Did you notice something unusual in the statistics above?**

All `docks_available` values are **zero**! This is not a data quality issue—it's a **fundamental characteristic** of the OV-fiets system that differs from traditional dock-based bike sharing systems (like Citi Bike, Santander Cycles, or Vélib).

#### 🚴 How OV-fiets Works:

1. **No docking system**: OV-fiets bikes must return to the **same station** where they were rented
2. **After-hours flexibility**: Bikes can be left outside the station after hours without being "docked"
3. **Capacity = Bikes**: Unlike dock-based systems, the "capacity" reflects the number of bikes, not physical docking infrastructure

#### 📊 Impact on Your Analysis and Modeling:

This domain knowledge will significantly affect your approach throughout the course:

**Module 3 (Data Exploration & Profiling):**
- ✅ Don't expect dock utilization patterns
- ✅ Focus on bike availability, not dock availability
- ✅ Station "capacity" represents bike inventory, not physical infrastructure

**Module 4 (Feature Engineering):**
- ❌ **Cannot create dock-based features** (e.g., "docks_full_ratio", "dock_pressure")
- ✅ **Focus on bike-centric features** (e.g., "bikes_per_station_hour", "zero_bike_frequency")
- ✅ Consider time-based patterns (commuter peaks, after-hours returns)

**Module 5 (Modeling):**
- ❌ Models predicting "dock availability" are not applicable
- ✅ **Target variable**: Predict bike availability at specific stations
- ✅ **Assumption**: Station capacity is relatively fixed (bike inventory)
- ✅ Consider station-to-station flow constraints (bikes must return to origin)

**Module 6 (Validation & Governance):**
- ✅ Validate that your model doesn't use dock-related features
- ✅ Ensure predictions respect the "same-station return" constraint
- ✅ Consider business rules: bikes_available ≤ station capacity

#### 🎯 Key Takeaway:

**Domain knowledge shapes data science!** Understanding how OV-fiets differs from traditional bike-sharing systems will:
- Prevent wasted effort on irrelevant features
- Guide appropriate modeling approaches
- Ensure your predictions are operationally valid

**💪 This is exactly why data exploration matters!** Always inspect your data carefully—the patterns (and absence of patterns) tell you how the system actually works.

---

### Task 8.2 Solution: Create 4-Panel Visualization Dashboard

In [ ]:
# SOLUTION: Task 8.2 - Create 4-panel visualization dashboard

# Create 2x2 subplot figure
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Bike Availability Dashboard', fontsize=16, fontweight='bold')

# Panel 1: Histogram of bikes_available
axes[0, 0].hist(df_bikes['bikes_available'], bins=20, color='steelblue', 
                edgecolor='black', alpha=0.7)
mean_bikes = df_bikes['bikes_available'].mean()
axes[0, 0].axvline(mean_bikes, color='red', linestyle='--', linewidth=2, 
                    label=f'Mean: {mean_bikes:.1f}')
axes[0, 0].set_xlabel('Bikes Available')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Available Bikes')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Panel 2: Histogram of utilization_pct
axes[0, 1].hist(df_bikes['utilization_pct'], bins=20, color='darkorange', 
                edgecolor='black', alpha=0.7)
mean_util = df_bikes['utilization_pct'].mean()
axes[0, 1].axvline(mean_util, color='red', linestyle='--', linewidth=2,
                    label=f'Mean: {mean_util:.1f}%')
axes[0, 1].set_xlabel('Utilization (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Station Utilization')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Panel 3: Top 10 stations by bikes_available
top_10_bikes = df_bikes.nlargest(10, 'bikes_available')
axes[1, 0].barh(range(len(top_10_bikes)), top_10_bikes['bikes_available'], 
                color='seagreen', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_10_bikes)))
axes[1, 0].set_yticklabels(top_10_bikes['name'], fontsize=9)
axes[1, 0].set_xlabel('Bikes Available')
axes[1, 0].set_title('Top 10 Stations by Available Bikes')
axes[1, 0].invert_yaxis()  # Highest at top
axes[1, 0].grid(axis='x', alpha=0.3)

# Panel 4: Average bikes vs docks
avg_bikes = df_bikes['bikes_available'].mean()
avg_docks = df_bikes['docks_available'].mean()
bars = axes[1, 1].bar(['Bikes Available', 'Docks Available'], 
                       [avg_bikes, avg_docks],
                       color=['steelblue', 'coral'], edgecolor='black')
axes[1, 1].set_ylabel('Average Count')
axes[1, 1].set_title('Average Station Availability')
axes[1, 1].grid(axis='y', alpha=0.3)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ 4-panel dashboard created!")

### Task 8.3 Solution: Create Geographic Station Map

In [ ]:
# SOLUTION: Task 8.3 - Create geographic station map

# Create figure
plt.figure(figsize=(12, 8))

# Prepare data for plotting - drop rows with missing coordinates or zero capacity
df_plot = df_bikes[df_bikes['total_capacity'] > 0].copy()
df_plot = df_plot.dropna(subset=['longitude', 'latitude', 'total_capacity'])

if len(df_plot) == 0:
    print("⚠️ No valid stations with capacity data for geographic map")
else:
    # Create scatter plot with color and size mapping
    scatter = plt.scatter(
        df_plot['longitude'], 
        df_plot['latitude'],
        c=df_plot['bikes_available'],  # Color by bikes available
        s=df_plot['total_capacity'] * 5,  # Size by capacity
        cmap='RdYlGn',  # Red=low, green=high
        edgecolors='black',
        linewidth=0.5,
        alpha=0.7
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, label='Bikes Available')
    
    # Find and annotate largest station
    if len(df_plot) > 0:
        largest_station = df_plot.nlargest(1, 'total_capacity').iloc[0]
        plt.annotate(
            f"{largest_station['name']}\n(Capacity: {int(largest_station['total_capacity'])})",
            xy=(largest_station['longitude'], largest_station['latitude']),
            xytext=(10, 10),
            textcoords='offset points',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color='black'),
            fontsize=9,
            fontweight='bold'
        )
    
    # Formatting
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.title('Geographic Distribution of Bike Stations\n(Size = Capacity, Color = Bikes Available)', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Geographic map created with {len(df_plot)} stations!")

### Task 8.4 Solution: Custom Analysis (Example)

**Example Analysis:** Station Utilization Categories

This example shows how to categorize stations and visualize the distribution of problematic vs healthy stations.

In [ ]:
# SOLUTION: Task 8.4 - Custom analysis example
# Analysis: Station Utilization Categories

# Categorize stations
def categorize_station(row):
    """Categorize station based on availability."""
    if row['bikes_available'] == 0:
        return 'Empty (No Bikes)'
    elif row['docks_available'] == 0:
        return 'Full (No Docks)'
    elif row['utilization_pct'] < 25:
        return 'Low Utilization (<25%)'
    elif row['utilization_pct'] > 75:
        return 'High Utilization (>75%)'
    else:
        return 'Healthy (25-75%)'

df_bikes['station_status'] = df_bikes.apply(categorize_station, axis=1)

# Count by category
category_counts = df_bikes['station_status'].value_counts()

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
colors = ['#ff6b6b', '#ffd93d', '#95e1d3', '#6bcf7f', '#4d96ff']
ax1.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 10})
ax1.set_title('Station Status Distribution', fontsize=14, fontweight='bold')

# Bar chart with counts
bars = ax2.bar(range(len(category_counts)), category_counts.values, 
               color=colors, edgecolor='black')
ax2.set_xticks(range(len(category_counts)))
ax2.set_xticklabels(category_counts.index, rotation=45, ha='right', fontsize=10)
ax2.set_ylabel('Number of Stations')
ax2.set_title('Station Count by Status', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("\\n📊 Station Status Summary:")
print("=" * 60)
for status, count in category_counts.items():
    pct = (count / len(df_bikes)) * 100
    print(f"{status:30s}: {count:3d} stations ({pct:5.1f}%)")
print("=" * 60)

# Calculate percentage of problematic stations
problematic = category_counts.get('Empty (No Bikes)', 0) + category_counts.get('Full (No Docks)', 0)
problematic_pct = (problematic / len(df_bikes)) * 100

print(f"\\n⚠️ Problematic stations (empty or full): {problematic} ({problematic_pct:.1f}%)")
print(f"✅ Healthy stations: {len(df_bikes) - problematic} ({100-problematic_pct:.1f}%)")

print("\\n✅ Custom analysis complete!")

---

### 🤔 Critical Thinking: Does This Analysis Make Sense?

**Look at the visualization above carefully!** Notice anything unusual about the "Full (No Docks)" category?

#### ❌ The Problem:

The categorization function checks `if row['docks_available'] == 0` to identify "full" stations. But remember from Task 8.1: **ALL docks_available values are 0** for OV-fiets! This means:

- Every single station is being marked as "Full (No Docks)"
- This category is meaningless for a system without docks
- The analysis conflates "no docking system" with "station full"

#### 💡 The Lesson:

**Domain knowledge affects every part of analysis!** A categorization that works for Citi Bike or Vélib doesn't work for OV-fiets. You must:

1. **Question your assumptions**: Does "Full (No Docks)" make sense here?
2. **Review your data**: Check what values actually exist (all zeros!)
3. **Adapt your approach**: Create categories appropriate for the system

#### ✅ Better Approach for OV-fiets:

Let's create a categorization based on **bike availability** only, which is what actually matters for this system:

In [ ]:
# CORRECTED ANALYSIS: Bike-centric categories for OV-fiets
# This approach recognizes that docks don't exist in this system

def categorize_station_ov_fiets(row):
    """
    Categorize station based on BIKE availability (appropriate for OV-fiets).
    No dock-based logic since OV-fiets doesn't use docks.
    """
    bikes = row['bikes_available']
    capacity = row['total_capacity']
    
    if bikes == 0:
        return 'Empty (No Bikes)'
    elif capacity > 0:
        pct_filled = (bikes / capacity) * 100
        if pct_filled < 20:
            return 'Critical Low (<20%)'
        elif pct_filled < 40:
            return 'Low (20-40%)'
        elif pct_filled < 70:
            return 'Good (40-70%)'
        else:
            return 'Well Stocked (>70%)'
    else:
        return 'Unknown'

# Apply corrected categorization
df_bikes['station_status_corrected'] = df_bikes.apply(categorize_station_ov_fiets, axis=1)

# Count by category
category_counts_corrected = df_bikes['station_status_corrected'].value_counts()

# Create corrected visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart with better color scheme for availability
colors_corrected = ['#ff4444', '#ff9933', '#ffd93d', '#66cc66', '#3399ff']
ax1.pie(category_counts_corrected.values, labels=category_counts_corrected.index, 
        autopct='%1.1f%%', colors=colors_corrected[:len(category_counts_corrected)], 
        startangle=90, textprops={'fontsize': 10})
ax1.set_title('Corrected: Station Availability Distribution\\n(Based on Bike Count Only)', 
              fontsize=14, fontweight='bold')

# Bar chart with counts
bars = ax2.bar(range(len(category_counts_corrected)), category_counts_corrected.values, 
               color=colors_corrected[:len(category_counts_corrected)], edgecolor='black')
ax2.set_xticks(range(len(category_counts_corrected)))
ax2.set_xticklabels(category_counts_corrected.index, rotation=45, ha='right', fontsize=10)
ax2.set_ylabel('Number of Stations')
ax2.set_title('Corrected: Station Count by Availability', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print corrected summary
print("\\n📊 Corrected Station Availability Summary:")
print("=" * 60)
for status, count in category_counts_corrected.items():
    pct = (count / len(df_bikes)) * 100
    print(f"{status:30s}: {count:3d} stations ({pct:5.1f}%)")
print("=" * 60)

# More meaningful metric: stations with critical/low availability
critical_low = (category_counts_corrected.get('Empty (No Bikes)', 0) + 
                category_counts_corrected.get('Critical Low (<20%)', 0))
critical_low_pct = (critical_low / len(df_bikes)) * 100

print(f"\\n⚠️ Stations needing attention (empty or <20%): {critical_low} ({critical_low_pct:.1f}%)")
print(f"✅ Stations with adequate bikes (>20%): {len(df_bikes) - critical_low} ({100-critical_low_pct:.1f}%)")

print("\\n✅ Corrected analysis complete!")
print("\\n💡 Key takeaway: Always validate that your analysis logic matches the domain!")

---

## 📚 Key Takeaways

### From Task 4.1-4.2 (API Basics - Part 4)
- ✅ APIs return structured data (usually JSON)
- ✅ Always check status codes (200 = success)
- ✅ Explore JSON structure before DataFrame conversion

### From Task 5.1 (Error Handling - Part 5)
- ✅ Use specific exception types
- ✅ Order exceptions from specific to general
- ✅ Return None for easy error checking
- ✅ Always include timeout parameter

### From Tasks 7.1-7.5 (DataFrame Operations - Part 7)
- ✅ pd.DataFrame() directly converts list of dicts
- ✅ Parse timestamps with pd.to_datetime()
- ✅ Vectorized operations are fast and clean
- ✅ Derived columns add analytical value

### From Tasks 8.1-8.4 (Visualization - Part 8)
- ✅ Start with summary statistics (.describe())
- ✅ Identify problem areas (empty/full stations)
- ✅ Multiple visualizations tell complete story

---

**🎉 Congratulations! You now have production-ready API data acquisition skills!**

**Next Steps:**
- M2_02: Weather Data API
- M2_03: Data Storage Best Practices
- M2_04: Merge Datasets

---

## 🔥 Part 9: Advanced Challenges - Solutions

**For advanced learners:** These solutions demonstrate professional-level patterns and best practices.

### Challenge 9.1 Solution: Retry Decorator with Exponential Backoff

In [ ]:
# SOLUTION: Challenge 9.1 - Retry decorator with exponential backoff

import functools
import time

def retry_on_failure(max_retries=3, backoff_factor=2):
    """
    Decorator that retries a function on failure with exponential backoff.
    
    Parameters:
    -----------
    max_retries : int
        Maximum number of retry attempts
    backoff_factor : int
        Multiplier for wait time between retries (exponential backoff)
    
    Returns:
    --------
    function
        Decorated function with retry logic
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # First attempt (not a retry)
            try:
                print(f"🔄 Attempt 1/{max_retries + 1}: Calling {func.__name__}...")
                return func(*args, **kwargs)
            except requests.exceptions.HTTPError as e:
                # Don't retry on 404 or other client errors (4xx)
                if e.response.status_code >= 400 and e.response.status_code < 500:
                    print(f"❌ Client error {e.response.status_code} - not retrying")
                    raise
                # Allow retries for server errors (5xx)
                last_exception = e
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                # Retry on timeout and connection errors
                last_exception = e
            
            # Retry logic
            for attempt in range(1, max_retries + 1):
                wait_time = backoff_factor ** attempt
                print(f"⏳ Waiting {wait_time} seconds before retry {attempt}/{max_retries}...")
                time.sleep(wait_time)
                
                try:
                    print(f"🔄 Attempt {attempt + 1}/{max_retries + 1}: Calling {func.__name__}...")
                    return func(*args, **kwargs)
                except requests.exceptions.HTTPError as e:
                    if e.response.status_code >= 400 and e.response.status_code < 500:
                        print(f"❌ Client error {e.response.status_code} - not retrying")
                        raise
                    last_exception = e
                except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                    last_exception = e
                    if attempt == max_retries:
                        print(f"❌ All {max_retries + 1} attempts failed")
                        raise
            
            # All retries exhausted
            print(f"❌ All {max_retries + 1} attempts failed")
            raise last_exception
        
        return wrapper
    return decorator


# Example usage
@retry_on_failure(max_retries=3, backoff_factor=2)
def fetch_bike_data_with_retry(network_id):
    """Fetch bike data with automatic retry on failure."""
    url = f"{BASE_URL}/networks/{network_id}"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


# Test with valid network
print("🧪 Test 1: Valid network (should succeed on first attempt)")
print("=" * 60)
try:
    data = fetch_bike_data_with_retry(NETWORK_ID)
    print(f"✅ Success! Fetched {len(data['network']['stations'])} stations\n")
except Exception as e:
    print(f"❌ Failed: {e}\n")

# Test with invalid network (should fail without retries - 404 is a client error)
print("\n🧪 Test 2: Invalid network (should fail immediately - no retries)")
print("=" * 60)
try:
    data = fetch_bike_data_with_retry("fake-network-xyz")
    print(f"✅ Success!\n")
except Exception as e:
    print(f"❌ Expected failure: {type(e).__name__}\n")

print("=" * 60)
print("✅ Retry decorator implemented successfully!")

### Challenge 9.2 Solution: Real-time Monitoring Function

**Note:** This is a simplified version for demonstration. In Jupyter, use `time.sleep()` for intervals. For production, use proper async/threading.

In [ ]:
# SOLUTION: Challenge 9.2 - Real-time monitoring function

def monitor_bikes(network_id, duration_seconds, interval=60):
    """
    Monitor bike availability over time with live plotting.
    
    Parameters:
    -----------
    network_id : str
        Network ID to monitor
    duration_seconds : int
        Total monitoring duration in seconds
    interval : int
        Seconds between each data fetch (default: 60)
    """
    import matplotlib.pyplot as plt
    from datetime import datetime
    
    # Enable interactive mode
    plt.ion()
    
    # Initialize data storage
    timestamps = []
    total_bikes = []
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    line, = ax.plot([], [], 'b-o', linewidth=2, markersize=8)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Total Bikes Available', fontsize=12)
    ax.set_title(f'🚴 Real-time Bike Availability Monitor: {network_id}', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    print(f"🔴 Starting monitoring for {duration_seconds}s (interval: {interval}s)")
    print(f"Press Ctrl+C to stop early\n")
    
    start_time = time.time()
    fetch_count = 0
    
    try:
        while (time.time() - start_time) < duration_seconds:
            try:
                # Fetch data
                data = fetch_bike_data_safe(network_id, timeout=10)
                
                if data:
                    # Calculate total bikes
                    stations = data['network']['stations']
                    bikes = sum(station['free_bikes'] for station in stations)
                    
                    # Store data
                    now = datetime.now()
                    timestamps.append(now)
                    total_bikes.append(bikes)
                    fetch_count += 1
                    
                    # Update plot
                    line.set_data(range(len(timestamps)), total_bikes)
                    ax.set_xlim(-0.5, len(timestamps) + 0.5)
                    ax.set_ylim(min(total_bikes) - 10, max(total_bikes) + 10)
                    
                    # Update x-tick labels with times
                    if len(timestamps) <= 10:
                        ax.set_xticks(range(len(timestamps)))
                        ax.set_xticklabels([t.strftime('%H:%M:%S') for t in timestamps], 
                                          rotation=45, ha='right')
                    
                    # Redraw
                    fig.canvas.draw()
                    fig.canvas.flush_events()
                    
                    print(f"📊 [{now.strftime('%H:%M:%S')}] Total bikes: {bikes} "
                          f"(Fetch #{fetch_count})")
                
                # Wait for next interval
                time.sleep(interval)
                
            except KeyboardInterrupt:
                print("\n⏸️ Monitoring stopped by user")
                break
                
    except KeyboardInterrupt:
        print("\n⏸️ Monitoring stopped by user")
    
    # Final summary
    elapsed = time.time() - start_time
    print("\n" + "=" * 60)
    print(f"📈 Monitoring Summary:")
    print(f"   Duration: {elapsed:.1f} seconds")
    print(f"   Data points collected: {fetch_count}")
    if total_bikes:
        print(f"   Min bikes: {min(total_bikes)}")
        print(f"   Max bikes: {max(total_bikes)}")
        print(f"   Avg bikes: {sum(total_bikes)/len(total_bikes):.1f}")
    print("=" * 60)
    
    # Disable interactive mode
    plt.ioff()
    plt.show()
    
    return timestamps, total_bikes


# Example: Monitor for 3 minutes with 30-second intervals
# Uncomment to run (warning: will take 3 minutes!)
# timestamps, bikes = monitor_bikes(NETWORK_ID, duration_seconds=180, interval=30)

print("✅ Real-time monitoring function implemented!")
print("⚠️ Uncomment the example to run (will take several minutes)")

### Challenge 9.3 Solution: Multi-Network Comparison

In [ ]:
# SOLUTION: Challenge 9.3 - Multi-network comparison

# Define networks to compare
networks = [
    {'id': 'ov-fiets', 'name': 'OV-fiets (Amsterdam)', 'city': 'Amsterdam'},
    {'id': 'velib', 'name': 'Vélib (Paris)', 'city': 'Paris'},
    {'id': 'santander-cycles', 'name': 'Santander Cycles (London)', 'city': 'London'}
]

# Fetch data from all networks
network_data = []

print("🌍 Fetching data from multiple networks...")
print("=" * 60)

for network in networks:
    print(f"\n📡 Fetching {network['name']}...")
    data = fetch_bike_data_safe(network['id'], timeout=15)
    
    if data:
        stations = data['network']['stations']
        total_stations = len(stations)
        total_bikes = sum(s['free_bikes'] for s in stations)
        # Handle None values for empty_slots (some APIs don't provide this)
        total_docks = sum(s['empty_slots'] if s['empty_slots'] is not None else 0 for s in stations)
        total_capacity = total_bikes + total_docks
        
        network_info = {
            'network_id': network['id'],
            'network_name': network['name'],
            'city': network['city'],
            'total_stations': total_stations,
            'total_bikes': total_bikes,
            'total_docks': total_docks,
            'total_capacity': total_capacity,
            'avg_bikes_per_station': total_bikes / total_stations if total_stations > 0 else 0,
            'avg_utilization': (total_bikes / total_capacity * 100) if total_capacity > 0 else 0
        }
        
        network_data.append(network_info)
        print(f"✅ Success! {total_stations} stations, {total_bikes} bikes")
    else:
        print(f"❌ Failed to fetch data")

print("\n" + "=" * 60)

# Create comparison DataFrame
df_comparison = pd.DataFrame(network_data)

# Display comparison table
print("\n📊 Network Comparison Summary:")
print("=" * 60)
display(df_comparison[['city', 'total_stations', 'total_bikes', 
                       'avg_bikes_per_station', 'avg_utilization']])

# Create 3-panel visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Multi-Network Bike Sharing Comparison', fontsize=16, fontweight='bold')

# Panel 1: Total Stations
axes[0].bar(df_comparison['city'], df_comparison['total_stations'], 
            color=['steelblue', 'crimson', 'forestgreen'], edgecolor='black')
axes[0].set_ylabel('Number of Stations')
axes[0].set_title('Total Stations by City')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_comparison['total_stations']):
    axes[0].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

# Panel 2: Average Bikes per Station
axes[1].bar(df_comparison['city'], df_comparison['avg_bikes_per_station'].round(1), 
            color=['steelblue', 'crimson', 'forestgreen'], edgecolor='black')
axes[1].set_ylabel('Average Bikes per Station')
axes[1].set_title('Avg Bikes per Station')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_comparison['avg_bikes_per_station']):
    axes[1].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Panel 3: Average Utilization
axes[2].bar(df_comparison['city'], df_comparison['avg_utilization'].round(1), 
            color=['steelblue', 'crimson', 'forestgreen'], edgecolor='black')
axes[2].set_ylabel('Utilization (%)')
axes[2].set_title('Average Utilization %')
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(50, color='red', linestyle='--', alpha=0.5, label='50% target')
axes[2].legend()
for i, v in enumerate(df_comparison['avg_utilization']):
    axes[2].text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Save combined data
comparison_file = DATA_DIR / f'multi_network_comparison_{TIMESTAMP}.csv'
df_comparison.to_csv(comparison_file, index=False)
print(f"\n💾 Saved comparison data: {comparison_file}")

print("\n✅ Multi-network comparison complete!")

### Challenge 9.4 Solution: API Response Caching

In [ ]:
# SOLUTION: Challenge 9.4 - API response caching

from datetime import datetime, timedelta

# Global cache dictionary
_api_cache = {}

def get_bike_data_cached(network_id, cache_duration=5, timeout=10):
    """
    Fetch bike data with intelligent caching.
    
    Parameters:
    -----------
    network_id : str
        Network ID to fetch
    cache_duration : int
        Cache validity duration in minutes (default: 5)
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        Cached or fresh data from API
    """
    now = datetime.now()
    
    # Check if we have cached data for this network
    if network_id in _api_cache:
        cached_data, cached_time = _api_cache[network_id]
        
        # Calculate cache age
        cache_age = (now - cached_time).total_seconds() / 60  # in minutes
        
        # Return cached data if still valid
        if cache_age < cache_duration:
            print(f"📦 Using cached data (age: {cache_age:.1f} min, "
                  f"expires in: {cache_duration - cache_age:.1f} min)")
            return cached_data
        else:
            print(f"⏰ Cache expired (age: {cache_age:.1f} min > {cache_duration} min)")
    
    # Fetch new data
    print(f"📡 Fetching new data from API...")
    data = fetch_bike_data_safe(network_id, timeout=timeout)
    
    if data:
        # Store in cache
        _api_cache[network_id] = (data, now)
        print(f"✅ Data fetched and cached (valid for {cache_duration} min)")
    
    return data


def clear_cache(network_id=None):
    """Clear cache for specific network or all networks."""
    global _api_cache
    if network_id:
        if network_id in _api_cache:
            del _api_cache[network_id]
            print(f"🗑️ Cache cleared for {network_id}")
    else:
        _api_cache = {}
        print(f"🗑️ All cache cleared")


def get_cache_status():
    """Display current cache status."""
    if not _api_cache:
        print("📭 Cache is empty")
        return
    
    print("📊 Cache Status:")
    print("=" * 60)
    now = datetime.now()
    for network_id, (data, cached_time) in _api_cache.items():
        age = (now - cached_time).total_seconds() / 60
        print(f"  {network_id:20s} - Age: {age:.1f} min")
    print("=" * 60)


# Test caching system
print("🧪 Testing cache system...")
print("=" * 60)

# First call - should fetch from API
print("\n🧪 Test 1: First call (should fetch from API)")
data1 = get_bike_data_cached(NETWORK_ID, cache_duration=5)

# Second call immediately - should use cache
print("\n🧪 Test 2: Immediate second call (should use cache)")
data2 = get_bike_data_cached(NETWORK_ID, cache_duration=5)

# Verify same data
print(f"\n✅ Data identical: {data1 == data2}")

# Check cache status
print()
get_cache_status()

# Clear cache
print()
clear_cache()

# Third call after clearing - should fetch from API again
print("\n🧪 Test 3: After clearing cache (should fetch from API)")
data3 = get_bike_data_cached(NETWORK_ID, cache_duration=5)

print("\n" + "=" * 60)
print("✅ Caching system implemented successfully!")
print("\n💡 Benefits:")
print("  - Reduces API calls and server load")
print("  - Faster response for repeated requests")
print("  - Respects cache duration for freshness")
print("  - Per-network caching with global cache management")